# EcoSort - Entraînement YOLO26 sur Kaggle (GPU T4)

Ce notebook exécute **le même pipeline DVC** qu'en local :
1. clone du dépôt GitHub ;
2. `dvc pull` des données préparées depuis DagsHub ;
3. `dvc repro -s train` avec suivi **MLflow sur DagsHub** ;
4. `dvc push` du modèle et export des fichiers à rapatrier en local (`kaggle_outputs.zip`).

**Prérequis Kaggle :** *Settings → Accelerator : GPU T4*, *Internet : On*, et un secret `DAGSHUB_TOKEN` (*Add-ons → Secrets*) attaché à ce notebook.

In [ ]:
# 1. Configuration
GITHUB_USER = "z-rayen"      # propriétaire du dépôt GitHub
DAGSHUB_USER = "z-rayen"     # ton nom d'utilisateur DagsHub (si différent, à modifier)
REPO = "ecosort"
BRANCH = "main"
WORKDIR = "/tmp/ecosort"     # hors de /kaggle/working : les données ne polluent pas les sorties
OUTPUT_DIR = "/kaggle/working"

In [ ]:
# 2. Vérification du GPU
!nvidia-smi

In [ ]:
# 3. Secrets et variables MLflow (le token n'est jamais affiché)
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/{REPO}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = token
print("MLflow ->", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
# 4. Clone du dépôt
!rm -rf {WORKDIR}
!git clone --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git {WORKDIR}
%cd {WORKDIR}
!git log --oneline -1

In [ ]:
# 5. Installation (torch CUDA est déjà fourni par Kaggle)
!pip install -q -r requirements/train.txt
!pip install -q -e .
!python -c "import torch, ultralytics; print(torch.__version__, torch.cuda.is_available(), ultralytics.__version__)"

In [ ]:
# 6. Identifiants DVC (écrits dans .dvc/config.local, jamais commités) + récupération des données
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user {DAGSHUB_USER}
!dvc remote modify origin --local password {token}
!dvc pull -r origin prepare
!cat data/processed/dataset.yaml
!echo "train:" $(ls data/processed/images/train | wc -l) "| val:" $(ls data/processed/images/val | wc -l) "| test:" $(ls data/processed/images/test | wc -l)

In [ ]:
# 7. Entraînement complet via DVC (stage `train` uniquement, sans relancer download/prepare)
!dvc repro -s train

In [ ]:
# 8. Envoi du modèle vers DagsHub + métriques
!dvc push -r origin train
!cat reports/train_metrics.json
!cat models/run_info.json

In [ ]:
# 9. Fichiers à rapatrier en local (Output -> kaggle_outputs.zip)
import shutil
import zipfile
from pathlib import Path

from IPython.display import Image, display

files = ["dvc.lock", "reports/train_metrics.json", "models/run_info.json", "models/.gitignore"]
with zipfile.ZipFile(f"{OUTPUT_DIR}/kaggle_outputs.zip", "w") as zf:
    for f in files:
        if Path(f).exists():
            zf.write(f, arcname=f)
            print("[OK] ajouté :", f)
        else:
            print("[WARN] absent :", f)

for name in ("results.png", "confusion_matrix_normalized.png"):
    src = Path("runs/train") / name
    if src.exists():
        shutil.copy(src, Path(OUTPUT_DIR) / name)
        display(Image(filename=str(src), width=900))